# Week 4 – Baseline Machine Learning Models  
## Predictive Maintenance for IT Infrastructure  

In this notebook, we build baseline machine learning models to predict server failure.

Objectives:
- Split dataset into training and testing sets  
- Train multiple ML models  
- Evaluate performance using standard metrics  
- Compare results across models  

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Load Dataset (Week 3 Output)

In [2]:
# Load feature engineered dataset
df = pd.read_csv("feature_engineered_dataset.csv")

df.head()

,CPU_Usage(%),Memory_Usage(%),Disk_IO(MB/s),Network_Latency(ms),Error_Rate,Downtime,failure,cpu_rolling_mean,memory_rolling_mean,cpu_lag1,memory_lag1,high_cpu_flag
0,-0.515745,-0.877662,-0.309787,-0.035849,1.319157,-0.962888,0,0.260266,-0.061869,0.735972,-0.819548,0
1,0.493351,0.780398,2.542629,-0.361470,-1.360802,-0.962888,0,0.417576,-0.148243,-0.515745,-0.877662,0
2,-0.510109,0.896258,0.784672,-0.367998,-1.007168,-0.962888,0,0.374190,0.193034,0.493351,0.780398,0
3,-0.512261,0.817024,0.386032,-1.455928,2.528054,1.038542,1,-0.061759,0.159294,-0.510109,0.896258,0
4,0.179479,-1.272365,-0.637582,-1.088707,0.045615,-0.962888,0,-0.173057,0.068730,-0.512261,0.817024,0


### Feature & Target Separation

In [3]:
# Define target
target = "failure"

# Separate features and target
X = df.drop(columns=["failure", "Downtime"])
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (9992, 10)
Target shape: (9992,)


### Train-Test Split (80/20)

In [4]:
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (7993, 10)
Testing data: (1999, 10)


## MODEL 1 – LOGISTIC REGRESSION

### Train Model

In [5]:
log_model = LogisticRegression()

log_model.fit(X_train, y_train)

LogisticRegression()

### Predictions

In [6]:
y_pred_log = log_model.predict(X_test)

### Evaluation

In [7]:
log_acc = accuracy_score(y_test, y_pred_log)
log_prec = precision_score(y_test, y_pred_log)
log_rec = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)

print("Logistic Regression Results:")
print("Accuracy:", log_acc)
print("Precision:", log_prec)
print("Recall:", log_rec)
print("F1 Score:", log_f1)

Logistic Regression Results:
Accuracy: 0.9489744872436218
Precision: 0.8222222222222222
Recall: 0.6788990825688074
F1 Score: 0.7437185929648241


## MODEL 2 – DECISION TREE

### Train Model

In [21]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

### Predictions & Evaluation

In [22]:
y_pred_dt = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_prec = precision_score(y_test, y_pred_dt)
dt_rec = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

print("\nDecision Tree Results:")
print("Accuracy:", dt_acc)
print("Precision:", dt_prec)
print("Recall:", dt_rec)
print("F1 Score:", dt_f1)


Decision Tree Results:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


## MODEL 3 – RANDOM FOREST

### Train Model

In [23]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Predictions & Evaluation

In [24]:
y_pred_rf = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf)
rf_rec = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("\nRandom Forest Results:")
print("Accuracy:", rf_acc)
print("Precision:", rf_prec)
print("Recall:", rf_rec)
print("F1 Score:", rf_f1)


Random Forest Results:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


## Performance Comparison Table

## Hyperparameter Tuning with Cross-Validation

In [25]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Define a stratified k-fold cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Logistic Regression Tuning

In [26]:
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid_search_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=cv, scoring='f1', n_jobs=-1)
grid_search_lr.fit(X_train, y_train)

best_lr = grid_search_lr.best_estimator_
y_pred_lr_tuned = best_lr.predict(X_test)

log_acc_tuned = accuracy_score(y_test, y_pred_lr_tuned)
log_prec_tuned = precision_score(y_test, y_pred_lr_tuned)
log_rec_tuned = recall_score(y_test, y_pred_lr_tuned)
log_f1_tuned = f1_score(y_test, y_pred_lr_tuned)

print("Tuned Logistic Regression Results:")
print("Best Parameters:", grid_search_lr.best_params_)
print("Accuracy:", log_acc_tuned)
print("Precision:", log_prec_tuned)
print("Recall:", log_rec_tuned)
print("F1 Score:", log_f1_tuned)

Tuned Logistic Regression Results:
Best Parameters: {'C': 1, 'solver': 'lbfgs'}
Accuracy: 0.9489744872436218
Precision: 0.8222222222222222
Recall: 0.6788990825688074
F1 Score: 0.7437185929648241


### Decision Tree Tuning

In [27]:
param_grid_dt = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=cv, scoring='f1', n_jobs=-1)
grid_search_dt.fit(X_train, y_train)

best_dt = grid_search_dt.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)

dt_acc_tuned = accuracy_score(y_test, y_pred_dt_tuned)
dt_prec_tuned = precision_score(y_test, y_pred_dt_tuned)
dt_rec_tuned = recall_score(y_test, y_pred_dt_tuned)
dt_f1_tuned = f1_score(y_test, y_pred_dt_tuned)

print("Tuned Decision Tree Results:")
print("Best Parameters:", grid_search_dt.best_params_)
print("Accuracy:", dt_acc_tuned)
print("Precision:", dt_prec_tuned)
print("Recall:", dt_rec_tuned)
print("F1 Score:", dt_f1_tuned)

Tuned Decision Tree Results:
Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


### Random Forest Tuning

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=cv, scoring='f1', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

best_rf = grid_search_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)

rf_acc_tuned = accuracy_score(y_test, y_pred_rf_tuned)
rf_prec_tuned = precision_score(y_test, y_pred_rf_tuned)
rf_rec_tuned = recall_score(y_test, y_pred_rf_tuned)
rf_f1_tuned = f1_score(y_test, y_pred_rf_tuned)

print("Tuned Random Forest Results:")
print("Best Parameters:", grid_search_rf.best_params_)
print("Accuracy:", rf_acc_tuned)
print("Precision:", rf_prec_tuned)
print("Recall:", rf_rec_tuned)
print("F1 Score:", rf_f1_tuned)

## Updated Performance Comparison Table (with Tuned Models)

In [20]:
results_tuned = pd.DataFrame({
    "Model": [
        "Logistic Regression (Untuned)",
        "Logistic Regression (Tuned)",
        "Decision Tree (Untuned)",
        "Decision Tree (Tuned)",
        "Random Forest (Untuned)",
        "Random Forest (Tuned)"
    ],
    "Accuracy": [
        log_acc,
        log_acc_tuned,
        dt_acc,
        dt_acc_tuned,
        rf_acc,
        rf_acc_tuned
    ],
    "Precision": [
        log_prec,
        log_prec_tuned,
        dt_prec,
        dt_prec_tuned,
        rf_prec,
        rf_prec_tuned
    ],
    "Recall": [
        log_rec,
        log_rec_tuned,
        dt_rec,
        dt_rec_tuned,
        rf_rec,
        rf_rec_tuned
    ],
    "F1 Score": [
        log_f1,
        log_f1_tuned,
        dt_f1,
        dt_f1_tuned,
        rf_f1,
        rf_f1_tuned
    ]
})

display(results_tuned)

results_tuned.to_csv("model_comparison_results_tuned.csv", index=False)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Untuned),0.948974,0.822222,0.678899,0.743719
1,Logistic Regression (Tuned),0.948974,0.822222,0.678899,0.743719


### Investigating Overfitting/Data Leakage

Given the perfect scores achieved by Decision Tree and Random Forest, we need to investigate potential overfitting or data leakage. This typically happens when the model learns the training data too well (overfitting) or when the training data contains information that is directly or indirectly predictive of the target, but would not be available in a real-world prediction scenario (data leakage).

In [ ]:
# Check the distribution of the target variable
print("Target variable distribution:\n", y.value_counts())
print("\nTarget variable percentage distribution:\n", y.value_counts(normalize=True))

# Check correlation of features with the target variable
# Since X is a numpy array after scaling, we need to convert it back to DataFrame for correlation with y
X_df = pd.DataFrame(X_train, columns=X.columns)
correlation_with_target = X_df.corrwith(y_train).sort_values(ascending=False)
print("\nCorrelation of features with target:\n", correlation_with_target)

Target variable distribution:
 failure
0    8927
1    1065
Name: count, dtype: int64

Target variable percentage distribution:
 failure
0    0.893415
1    0.106585
Name: proportion, dtype: float64

Correlation of features with target:
 Disk_IO(MB/s)          0.026568
cpu_rolling_mean       0.022340
cpu_lag1               0.013543
memory_lag1            0.012541
Network_Latency(ms)    0.009535
memory_rolling_mean    0.009493
Error_Rate             0.007607
high_cpu_flag          0.005882
CPU_Usage(%)           0.004565
Memory_Usage(%)       -0.011087
dtype: float64


### Feature Importance Analysis

Let's also look at the feature importances from the best-tuned Decision Tree and Random Forest models. This can highlight which features are dominating the prediction and potentially causing the perfect scores.

In [ ]:
# Feature importance for Tuned Decision Tree
print("\nDecision Tree (Tuned) Feature Importances:")
dt_feature_importance = pd.Series(best_dt.feature_importances_, index=X.columns).sort_values(ascending=False)
print(dt_feature_importance)

# Feature importance for Tuned Random Forest
print("\nRandom Forest (Tuned) Feature Importances:")
rf_feature_importance = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(rf_feature_importance)


Decision Tree (Tuned) Feature Importances:
Error_Rate             0.401386
Memory_Usage(%)        0.301875
high_cpu_flag          0.296740
CPU_Usage(%)           0.000000
Network_Latency(ms)    0.000000
Disk_IO(MB/s)          0.000000
cpu_rolling_mean       0.000000
memory_rolling_mean    0.000000
cpu_lag1               0.000000
memory_lag1            0.000000
dtype: float64

Random Forest (Tuned) Feature Importances:
Error_Rate             0.407859
Memory_Usage(%)        0.289132
high_cpu_flag          0.150974
CPU_Usage(%)           0.130137
memory_rolling_mean    0.007567
cpu_rolling_mean       0.004766
memory_lag1            0.003208
cpu_lag1               0.002697
Network_Latency(ms)    0.001874
Disk_IO(MB/s)          0.001787
dtype: float64


### Re-evaluating Models without Highly Predictive Features

Given the perfect scores, we will now remove the highly predictive features (`Error_Rate`, `Memory_Usage(%)`, `high_cpu_flag`) and re-evaluate the models to get a more realistic assessment of their performance on a less 'leaky' dataset. This will help us understand if the models can still perform well when forced to rely on other, potentially less direct, indicators of failure.

In [ ]:
# Define the features to be dropped
features_to_drop = ['Error_Rate', 'Memory_Usage(%)', 'high_cpu_flag']

# Create a new feature set by dropping these columns
X_revised = df.drop(columns=features_to_drop + [target, 'Downtime'])
y_revised = df[target]

print("Revised Features shape:", X_revised.shape)
print("Target shape:", y_revised.shape)

# Re-split the data with the revised feature set
X_train_revised, X_test_revised, y_train_revised, y_test_revised = train_test_split(
    X_revised, y_revised, test_size=0.2, random_state=42
)

# Re-scale the revised data
scaler_revised = StandardScaler()
scaler_revised.fit(X_train_revised)
X_train_revised = scaler_revised.transform(X_train_revised)
X_test_revised = scaler_revised.transform(X_test_revised)

print("\nRevised Training data:", X_train_revised.shape)
print("Revised Testing data:", X_test_revised.shape)

Revised Features shape: (9992, 7)
Target shape: (9992,)

Revised Training data: (7993, 7)
Revised Testing data: (1999, 7)


### MODEL 1 (REVISED) – LOGISTIC REGRESSION

In [ ]:
log_model_revised = LogisticRegression(random_state=42)
log_model_revised.fit(X_train_revised, y_train_revised)
y_pred_log_revised = log_model_revised.predict(X_test_revised)

log_acc_revised = accuracy_score(y_test_revised, y_pred_log_revised)
log_prec_revised = precision_score(y_test_revised, y_pred_log_revised)
log_rec_revised = recall_score(y_test_revised, y_pred_log_revised)
log_f1_revised = f1_score(y_test_revised, y_pred_log_revised)

print("Logistic Regression (Revised) Results:")
print("Accuracy:", log_acc_revised)
print("Precision:", log_prec_revised)
print("Recall:", log_rec_revised)
print("F1 Score:", log_f1_revised)

Logistic Regression (Revised) Results:
Accuracy: 0.896448224112056
Precision: 0.9230769230769231
Recall: 0.05504587155963303
F1 Score: 0.1038961038961039


### MODEL 2 (REVISED) – DECISION TREE

In [ ]:
dt_model_revised = DecisionTreeClassifier(random_state=42)
dt_model_revised.fit(X_train_revised, y_train_revised)
y_pred_dt_revised = dt_model_revised.predict(X_test_revised)

dt_acc_revised = accuracy_score(y_test_revised, y_pred_dt_revised)
dt_prec_revised = precision_score(y_test_revised, y_pred_dt_revised)
dt_rec_revised = recall_score(y_test_revised, y_pred_dt_revised)
dt_f1_revised = f1_score(y_test_revised, y_pred_dt_revised)

print("Decision Tree (Revised) Results:")
print("Accuracy:", dt_acc_revised)
print("Precision:", dt_prec_revised)
print("Recall:", dt_rec_revised)
print("F1 Score:", dt_f1_revised)

Decision Tree (Revised) Results:
Accuracy: 0.8564282141070535
Precision: 0.35684647302904565
Recall: 0.3944954128440367
F1 Score: 0.3747276688453159


### MODEL 3 (REVISED) – RANDOM FOREST

In [ ]:
rf_model_revised = RandomForestClassifier(random_state=42)
rf_model_revised.fit(X_train_revised, y_train_revised)
y_pred_rf_revised = rf_model_revised.predict(X_test_revised)

rf_acc_revised = accuracy_score(y_test_revised, y_pred_rf_revised)
rf_prec_revised = precision_score(y_test_revised, y_pred_rf_revised)
rf_rec_revised = recall_score(y_test_revised, y_pred_rf_revised)
rf_f1_revised = f1_score(y_test_revised, y_pred_rf_revised)

print("Random Forest (Revised) Results:")
print("Accuracy:", rf_acc_revised)
print("Precision:", rf_prec_revised)
print("Recall:", rf_rec_revised)
print("F1 Score:", rf_f1_revised)

Random Forest (Revised) Results:
Accuracy: 0.9259629814907454
Precision: 0.9605263157894737
Recall: 0.3348623853211009
F1 Score: 0.4965986394557823


### Hyperparameter Tuning (REVISED MODELS)

#### Logistic Regression Tuning (Revised)

In [ ]:
grid_search_lr_revised = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=cv, scoring='f1', n_jobs=-1)
grid_search_lr_revised.fit(X_train_revised, y_train_revised)

best_lr_revised = grid_search_lr_revised.best_estimator_
y_pred_lr_tuned_revised = best_lr_revised.predict(X_test_revised)

log_acc_tuned_revised = accuracy_score(y_test_revised, y_pred_lr_tuned_revised)
log_prec_tuned_revised = precision_score(y_test_revised, y_pred_lr_tuned_revised)
log_rec_tuned_revised = recall_score(y_test_revised, y_pred_lr_tuned_revised)
log_f1_tuned_revised = f1_score(y_test_revised, y_pred_lr_tuned_revised)

print("Tuned Logistic Regression (Revised) Results:")
print("Best Parameters:", grid_search_lr_revised.best_params_)
print("Accuracy:", log_acc_tuned_revised)
print("Precision:", log_prec_tuned_revised)
print("Recall:", log_rec_tuned_revised)
print("F1 Score:", log_f1_tuned_revised)

Tuned Logistic Regression (Revised) Results:
Best Parameters: {'C': 10, 'solver': 'liblinear'}
Accuracy: 0.896448224112056
Precision: 0.9230769230769231
Recall: 0.05504587155963303
F1 Score: 0.1038961038961039


#### Decision Tree Tuning (Revised)

In [ ]:
grid_search_dt_revised = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=cv, scoring='f1', n_jobs=-1)
grid_search_dt_revised.fit(X_train_revised, y_train_revised)

best_dt_revised = grid_search_dt_revised.best_estimator_
y_pred_dt_tuned_revised = best_dt_revised.predict(X_test_revised)

dt_acc_tuned_revised = accuracy_score(y_test_revised, y_pred_dt_tuned_revised)
dt_prec_tuned_revised = precision_score(y_test_revised, y_pred_dt_tuned_revised)
dt_rec_tuned_revised = recall_score(y_test_revised, y_pred_dt_tuned_revised)
dt_f1_tuned_revised = f1_score(y_test_revised, y_pred_dt_tuned_revised)

print("Tuned Decision Tree (Revised) Results:")
print("Best Parameters:", grid_search_dt_revised.best_params_)
print("Accuracy:", dt_acc_tuned_revised)
print("Precision:", dt_prec_tuned_revised)
print("Recall:", dt_rec_tuned_revised)
print("F1 Score:", dt_f1_tuned_revised)

Tuned Decision Tree (Revised) Results:
Best Parameters: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10}
Accuracy: 0.9234617308654327
Precision: 0.8735632183908046
Recall: 0.3486238532110092
F1 Score: 0.49836065573770494


#### Random Forest Tuning (Revised)

In [ ]:
grid_search_rf_revised = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=cv, scoring='f1', n_jobs=-1)
grid_search_rf_revised.fit(X_train_revised, y_train_revised)

best_rf_revised = grid_search_rf_revised.best_estimator_
y_pred_rf_tuned_revised = best_rf_revised.predict(X_test_revised)

rf_acc_tuned_revised = accuracy_score(y_test_revised, y_pred_rf_tuned_revised)
rf_prec_tuned_revised = precision_score(y_test_revised, y_pred_rf_tuned_revised)
rf_rec_tuned_revised = recall_score(y_test_revised, y_pred_rf_tuned_revised)
rf_f1_tuned_revised = f1_score(y_test_revised, y_pred_rf_tuned_revised)

print("Tuned Random Forest (Revised) Results:")
print("Best Parameters:", grid_search_rf_revised.best_params_)
print("Accuracy:", rf_acc_tuned_revised)
print("Precision:", rf_prec_tuned_revised)
print("Recall:", rf_rec_tuned_revised)
print("F1 Score:", rf_f1_tuned_revised)

Tuned Random Forest (Revised) Results:
Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Accuracy: 0.9249624812406203
Precision: 0.9594594594594594
Recall: 0.3256880733944954
F1 Score: 0.4863013698630137


### Performance Comparison Table (Revised Features)

In [ ]:
results_revised = pd.DataFrame({
    "Model": [
        "Logistic Regression (Untuned - Original)", "Decision Tree (Untuned - Original)", "Random Forest (Untuned - Original)",
        "Logistic Regression (Tuned - Original)", "Decision Tree (Tuned - Original)", "Random Forest (Tuned - Original)",
        "Logistic Regression (Untuned - Revised)", "Decision Tree (Untuned - Revised)", "Random Forest (Untuned - Revised)",
        "Logistic Regression (Tuned - Revised)", "Decision Tree (Tuned - Revised)", "Random Forest (Tuned - Revised)"
    ],
    "Accuracy": [
        log_acc, dt_acc, rf_acc,
        log_acc_tuned, dt_acc_tuned, rf_acc_tuned,
        log_acc_revised, dt_acc_revised, rf_acc_revised,
        log_acc_tuned_revised, dt_acc_tuned_revised, rf_acc_tuned_revised
    ],
    "Precision": [
        log_prec, dt_prec, rf_prec,
        log_prec_tuned, dt_prec_tuned, rf_prec_tuned,
        log_prec_revised, dt_prec_revised, rf_prec_revised,
        log_prec_tuned_revised, dt_prec_tuned_revised, rf_prec_tuned_revised
    ],
    "Recall": [
        log_rec, dt_rec, rf_rec,
        log_rec_tuned, dt_rec_tuned, rf_rec_tuned,
        log_rec_revised, dt_rec_revised, rf_rec_revised,
        log_rec_tuned_revised, dt_rec_tuned_revised, rf_rec_tuned_revised
    ],
    "F1 Score": [
        log_f1, dt_f1, rf_f1,
        log_f1_tuned, dt_f1_tuned, rf_f1_tuned,
        log_f1_revised, dt_f1_revised, rf_f1_revised,
        log_f1_tuned_revised, dt_f1_tuned_revised, rf_f1_tuned_revised
    ]
})

display(results_revised)

results_revised.to_csv("model_comparison_results_revised.csv", index=False)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Untuned - Original),0.948974,0.822222,0.678899,0.743719
1,Decision Tree (Untuned - Original),1.000000,1.000000,1.000000,1.000000
2,Random Forest (Untuned - Original),1.000000,1.000000,1.000000,1.000000
3,Logistic Regression (Tuned - Original),0.948974,0.822222,0.678899,0.743719
4,Decision Tree (Tuned - Original),1.000000,1.000000,1.000000,1.000000
5,Random Forest (Tuned - Original),1.000000,1.000000,1.000000,1.000000
6,Logistic Regression (Untuned - Revised),0.896448,0.923077,0.055046,0.103896
7,Decision Tree (Untuned - Revised),0.856428,0.356846,0.394495,0.374728
8,Random Forest (Untuned - Revised),0.925963,0.960526,0.334862,0.496599
9,Logistic Regression (Tuned - Revised),0.896448,0.923077,0.055046,0.103896


In [ ]:
results_revised

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Untuned - Original),0.948974,0.822222,0.678899,0.743719
1,Decision Tree (Untuned - Original),1.000000,1.000000,1.000000,1.000000
2,Random Forest (Untuned - Original),1.000000,1.000000,1.000000,1.000000
3,Logistic Regression (Tuned - Original),0.948974,0.822222,0.678899,0.743719
4,Decision Tree (Tuned - Original),1.000000,1.000000,1.000000,1.000000
5,Random Forest (Tuned - Original),1.000000,1.000000,1.000000,1.000000
6,Logistic Regression (Untuned - Revised),0.896448,0.923077,0.055046,0.103896
7,Decision Tree (Untuned - Revised),0.856428,0.356846,0.394495,0.374728
8,Random Forest (Untuned - Revised),0.925963,0.960526,0.334862,0.496599
9,Logistic Regression (Tuned - Revised),0.896448,0.923077,0.055046,0.103896


### Save Results

In [ ]:
results_revised.to_csv("model_comparison_results.csv", index=False)

In [ ]:
from google.colab import files
files.download('model_comparison_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>